<a href="https://colab.research.google.com/github/qndks11/MNIST/blob/main/MNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torchvision import datasets, transforms

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_availbable() else 'cpu')

BATCH_SIZE = 32
EPOCHS = 25

# 데이터 다운로드하기

In [ ]:
# Define a transform to normalize the data
transform = transforms.Compose([transforms.ToTensor()])

# 트레이닝 데이터 다운로드
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

# 테스트 데이터 다운로드
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size = BATCH_SIZE, shuffle=True)

test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size = BATCH_SIZE, shuffle=True)

# CNN 모델

In [ ]:
class CNN(torch.nn.Module):
  def __init__(self):
    super(CNN, self).__init__()
    self.conv1 = torch.nn.Conv2d(in_channels=1, out_channels=16, kernel_size=5, stride=1)
    self.conv2 = torch.nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1)
    self.pool = torch.nn.MaxPool2d(kernel_size=2)
    self.relu = torch.nn.ReLU(inplace=True)

    self.fc1 = torch.nn.Linera(32*5*5, 64)
    self.fc2 = torch.nn.Lineara(64, 10)

  def forward(self, x):
    x = self.conv1(x)
    x = self.relu(x)
    x = self.pool(x)
    x = self.conv2(x)
    x = self.relu(x)
    x = self.pool(x)

    x = x.view(x.size(0), -1)
    x = self.fc1(x)
    x = self.relu(x)
    x = self.fc2(x)
    return x

# 트레이닝

In [ ]:
model = CNN().to(DEVICE)
criterion = torch.nn.CrossEntropyLoss
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.train()

for epoch in range(EPOCHS)
  running_loss = 0.0

  for images, labels in train_loader:
    imgaes, labels = images.to(DEVICE), labels.to(DEVICE)

    optimizer.zero_grad()       # 그레디언트 초기화
    outputs = model(images)     # Forward
    loss = criterion(outputs, labels)  # loss 계산
    loss.backward()             # backward
    optimizer.step()            # 가중치 업데이트

    running_loss += loss.item()

  avg_loss = running_loss / len(train_loader)
  print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {avg_loss:.4f}")

# 평가

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
  for images, labels in test_loader:
    images, labels = images.to(DEVICE), labels.to(DEVICE)
    outputs = model(images)
    _, predicted = torch.max(outputs, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item() # 맞춘 개수가 나옴

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")